# **Report 1: Analysing Matrix Monitoring Methods In Accelerating Frequent Pattern Recognition Algorithms**

### **Professor:** Dr. Ghatee

### **Head TA:** Behnam Yousefimehr

### **Author:** Hassan Hajizadeh

## **Step 1: Downloading, Loading And Preprocessing The Data From [UCI Website](https://archive.ics.uci.edu/dataset/352/online+retail)**

We already downloaded the Dataset and for properly loading the file we will install pandas and openpyxl (for loading xlsx) libraries.

We also install numpy library for calculations and scikit-learn for using its algorithm methods.

And we will install matplotlib library for plotting our data.

In [1]:
!pip install pandas
!pip install openpyxl
!pip install numpy
!pip install scikit-learn
!pip install matplotlib

### **Loading Data**

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel('Online Retail.xlsx')

### **Cleaning Data**

Based on what we see below in dataset info, indicates that there are some empty values in the description and the CustomerID Features.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


#### **Editing Or Dropping Bad Samples**

Based on the project description, we will just need the **InvoiceNo** and **StockCode** and **InvoiceDate** columns for the rest of the project.

So dropping almost **140,000** rows of data, just because of **ONE** column, the **CustomerID** column, Would Be unnecessary.

And also Based on the **StockCode** Description in UCI Dataset page in the website, mentions that each product is **"a 5-digit integral number uniquely assigned to each distinct product"** so we will drop the rows that their **StockCode** doesn't Start with a **5-digit integral number**.

In the mean time, we will remove the letter at the end of some **StockCodes** because it's just the color and type letter and we don't want to make another category for each color of a product too.

And we will also drop every row that their **InvoiceNo** is not just **a 6-digits number**. beacuse of description of **InvoiceNo** values that mentions it should be **"a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation."**

In [4]:
df['StockCode'] = df['StockCode'].astype(str).str.strip()
mask = df['StockCode'].str.match(r'^\d{5}', na=False)
df = df[mask].copy()
df['StockCode'] = df['StockCode'].str.extract(r'^(\d{5})')

df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()
mask = ~df['InvoiceNo'].str.fullmatch(r'\d{6}', na=False)
df = df.drop(df[mask].index)

#### **Checking And Removing The Negative Data**

In [5]:
(df[["Quantity","UnitPrice","CustomerID"]] < 0).any().any()

np.True_

In [6]:
df[df[["Quantity","UnitPrice","CustomerID"]] < 0].stack()

2406    Quantity     -10.0
4347    Quantity     -38.0
7188    Quantity     -20.0
7189    Quantity     -20.0
7190    Quantity      -6.0
                     ...  
535333  Quantity     -26.0
535335  Quantity   -1050.0
535336  Quantity     -30.0
536908  Quantity    -338.0
538919  Quantity    -235.0
Length: 1324, dtype: object

In [7]:
df = df[(df['Quantity'] >= 0) & (df['UnitPrice'] >= 0)]

In [8]:
(df[["Quantity","UnitPrice","CustomerID"]] < 0).any().any()

np.False_

**PS: By This Simple Tricks We Saved 130,962 Rows Of Data From Getting Dropped**

## **Step 2: Creating Transaction-Item Binary Matrix**

Based on matrix below we will use InvoiceNo and StockCode categories to create our binary matrix.

In [9]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [10]:
binary_matrix = pd.crosstab(df['InvoiceNo'], df['StockCode'])
binary_matrix = (binary_matrix > 0).astype(int)
binary_matrix


StockCode,10002,10080,10120,10123,10124,10125,10133,10135,11001,15030,...,90202,90204,90205,90206,90208,90209,90210,90211,90212,90214
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581584,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581585,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## **Step 3: Streaming Simulation For Binary Matrix**

For streaming simulation first we sort the binary matrix based on InvoiceDate.

In [11]:
first_time_for_each_transaction = df.groupby('InvoiceNo')['InvoiceDate'].min()
sorted_binary_matrix = binary_matrix.loc[first_time_for_each_transaction.sort_values().index]


Then we divide this sorted binary matrix to 14 batches.

**PS: I just choosed 14, to our batche sizes be equal togather.**

In [12]:
number_of_batches = 14
batch_size = int(np.ceil(len(sorted_binary_matrix) / number_of_batches))
batches = [
    sorted_binary_matrix.iloc[i*batch_size : (i+1)*batch_size]
    for i in range(number_of_batches)
]


## **Step 4: Matrix Monitoring Using Different Algorithms**

In this section we will use matrix skeching methods for increasing speed and decreasing memory usage as well as preserving main statistical information.

### **First Matrix Skeching Method: Gaussian Random Projection**

In [13]:
def generate_random_matrix(number_of_samples , original_dimension , new_dimension=None , eps=None):
    if(new_dimension == None and eps==None):
        raise ValueError("Both of new_dimension and eps couldn't be None togather")
    if (eps != None):
        new_dimension = int(4 * np.log(number_of_samples) / (eps**2 / 2 - eps**3 / 3))
    random_matrix = np.random.normal(0,1/np.sqrt(new_dimension),size=(original_dimension,new_dimension))
    return random_matrix
    
def gaussian_random_projection(matrix , random_matrix=None , eps=None , new_dimension=None):
    if (random_matrix == None):
        random_matrix = generate_random_matrix(number_of_samples=len(matrix), original_dimension=len(matrix.columns),new_dimension=new_dimension , eps=eps)
        
    return np.dot(matrix,random_matrix) , random_matrix # i didn't used seed and just returned random_matrix itself for the situations 
                                                        # that batch sizes aren't equal. (i did that for preventing miscalulation of new dimension)


### **Second Matrix Skeching Method: Incremental PCA**

In [14]:
import numpy as np

class IncrementalPCA:
    def __init__(self, n_components):
        self.mean_ = None
        self.cov_ = None
        self.n_components = n_components
        self.n_samples_seen = 0

    def partial_fit(self, X):
        X = np.asarray(X)
        n_new = X.shape[0]
        if self.mean_ is None:
            self.mean_ = np.mean(X, axis=0)
            self.cov_ = np.cov(X, rowvar=False) * (n_new - 1)
            self.n_samples_seen = n_new
        else:
            old_mean = self.mean_
            new_mean = np.mean(X, axis=0)
            total_n = self.n_samples_seen + n_new
            updated_mean = (self.n_samples_seen * old_mean + n_new * new_mean) / total_n
            X_centered = X - new_mean
            new_cov = X_centered.T @ X_centered
            mean_diff = (old_mean - new_mean).reshape(-1, 1)
            mean_correction = (self.n_samples_seen * n_new) / total_n * (mean_diff @ mean_diff.T)
            updated_cov = self.cov_ + new_cov + mean_correction
            self.mean_ = updated_mean
            self.cov_ = updated_cov
            self.n_samples_seen = total_n

    def components_(self):
        if self.cov_ is None:
            return None
        eigvals, eigvecs = np.linalg.eigh(self.cov_ / (self.n_samples_seen - 1))
        order = np.argsort(eigvals)[::-1]
        return eigvecs[:, order[:self.n_components]].T

    def transform(self, X):
        X = np.asarray(X)
        components = self.components_()
        if self.mean_ is not None:
            X_c = X - self.mean_
        else:
            X_c = X
        return X_c @ components.T

In [15]:
ipca = IncrementalPCA(n_components=500)
ipca_result = []
for i in range(len(batches)-1):
    ipca.partial_fit(batches[i])
    ipca_result.append(ipca.transform(batches[0]))

### **Third Matrix Skeching Method: Frequent Directions**